# 06 · Gráficos dos indicadores

Os gráficos são produzidos a partir de `Resultados_Indicadores_TCC_revisado.xlsx`. Cada célula salva um PNG na pasta `resultados/graficos`. A lista de figuras corresponde à entrega visual do TCC, incluindo os gráficos extras dos artigos criados e preexistentes.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
BRUTOS = ROOT / 'dados' / 'brutos'
PROCESSADOS = ROOT / 'dados' / 'processados'
GRAFICOS = ROOT / 'resultados' / 'graficos'
PROCESSADOS.mkdir(parents=True, exist_ok=True)
GRAFICOS.mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from matplotlib.patches import Patch
import textwrap

ARQUIVO = PROCESSADOS / 'Resultados_Indicadores_TCC_revisado.xlsx'
verbetes = pd.read_excel(ARQUIVO,sheet_name='Verbetes_70')
individuais = pd.read_excel(ARQUIVO,sheet_name='Participacoes_177')
grupos = pd.read_excel(ARQUIVO,sheet_name='Grupos_70')

DIM=['IC','IV','IEO','ICN','IRV']
NOMES=['Completude','Verificabilidade','Estrutura e organização','Conectividade','Recursos visuais']
CURTOS=['Comp.','Verif.','Estrut.','Conect.','Visuais']
AZUL='#4477AA'; LARANJA='#EE8866'; VERDE='#228833'; ROXO='#AA4499'; CINZA='#64748B'
RADAR=['#007F86','#648FFF','#D55E00','#7D4EAF','#C44569']
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,
    'axes.titlesize':13, 'axes.labelsize':10, 'savefig.facecolor':'white'})

def salvar(fig,nome):
    caminho=GRAFICOS/nome
    fig.savefig(caminho,dpi=220,bbox_inches='tight',facecolor='white')
    plt.close(fig)
    print('Salvo:',caminho.name)

def dividir(tamanho=25):
    return lambda x:'\n'.join(textwrap.wrap(str(x),width=tamanho,break_long_words=False,break_on_hyphens=False))

def barras_horizontais(df,rotulo,valor,titulo,nome,cor=AZUL,limite=5,complemento=''):
    a=df.sort_values(valor,ascending=False).head(15).iloc[::-1]
    altura=max(8,0.64*len(a)+2.5)
    fig,ax=plt.subplots(figsize=(12,altura))
    yy=np.arange(len(a))
    ax.barh(yy,a[valor],height=.68,color=cor,zorder=3)
    ax.set_yticks(yy,[dividir(31)(x) for x in a[rotulo]],fontsize=9.4)
    ax.set_xlim(0,limite)
    ax.set_xlabel(complemento or 'Pontuação da soma (0 a 5)')
    ax.set_title(titulo,pad=15,fontweight='bold')
    ax.grid(axis='x',color='#e5e7eb',zorder=0)
    ax.tick_params(axis='y',length=0,pad=8)
    for i,ponto in enumerate(a[valor]):
        ax.text(min(float(ponto)+.04,limite-.04),i,f'{ponto:.2f}',va='center',
                ha='left' if ponto<limite-.4 else 'right',fontsize=9)
    fig.subplots_adjust(left=.36,right=.96,bottom=.09,top=.93)
    salvar(fig,nome)

def radar(df,rotulo,campos,ordem,titulo,nome):
    a=df.nlargest(5,ordem)
    ang=np.linspace(0,2*np.pi,5,endpoint=False).tolist();ang+=[ang[0]]
    fig=plt.figure(figsize=(10.5,7.2))
    ax=fig.add_axes([.08,.12,.55,.73],projection='polar')
    for i,(_,r) in enumerate(a.iterrows()):
        valores=[float(r[c]) for c in campos]
        ax.plot(ang,valores+[valores[0]],color=RADAR[i],lw=2.5,label=dividir(29)(r[rotulo]))
        ax.fill(ang,valores+[valores[0]],color=RADAR[i],alpha=.07)
    # Siglas afastadas do contorno para não colidir com os valores do eixo.
    ax.set_thetagrids(np.rad2deg(ang[:-1]),DIM)
    ax.tick_params(axis='x',pad=22,labelsize=10)
    ax.set_ylim(0,1.04)
    ax.set_yticks([.25,.5,.75,1]);ax.set_yticklabels(['0,25','0,50','0,75','1,00'],fontsize=8)
    ax.set_rlabel_position(110)
    ax.grid(alpha=.25)
    fig.suptitle(titulo,y=.98,fontweight='bold',fontsize=15)
    legend=ax.legend(title='Verbetes / participações',frameon=False,loc='center left',
                     bbox_to_anchor=(1.33,.5),labelspacing=1.1,borderaxespad=0)
    legend._legend_box.align='left'
    fig.text(.08,.03,'IC: completude  |  IV: verificabilidade  |  IEO: estrutura e organização  |  ICN: conectividade  |  IRV: recursos visuais',fontsize=8.2,color=CINZA)
    salvar(fig,nome)

def heatmap(df,rotulo,campos,titulo,nome,diferencas=False):
    a=df.head(15).copy()
    valores=a[campos].to_numpy(float)
    fig,ax=plt.subplots(figsize=(10,10))
    if diferencas:
        lim=max(.2,np.max(np.abs(valores)))
        norm=mcolors.TwoSlopeNorm(vmin=-lim,vcenter=0,vmax=lim)
        im=ax.imshow(valores,cmap='RdBu',norm=norm,aspect='auto')
    else:
        im=ax.imshow(valores,cmap='YlGnBu',vmin=0,vmax=1,aspect='auto')
    ax.set_yticks(np.arange(len(a)),[dividir(32)(x) for x in a[rotulo]],fontsize=9)
    ax.set_xticks(np.arange(5),CURTOS)
    ax.set_title(titulo,fontweight='bold',pad=14)
    for i in range(valores.shape[0]):
        for j in range(5):
            rgba=im.cmap(im.norm(valores[i,j]))
            brilho=.2126*rgba[0]+.7152*rgba[1]+.0722*rgba[2]
            ax.text(j,i,f'{valores[i,j]:+.2f}' if diferencas else f'{valores[i,j]:.2f}',
                    ha='center',va='center',fontsize=8,color='black' if brilho>.55 else 'white')
    cb=fig.colorbar(im,ax=ax,shrink=.85,pad=.025)
    cb.set_label('Variação' if diferencas else 'Pontuação')
    fig.subplots_adjust(left=.34,right=.92,bottom=.06,top=.93)
    salvar(fig,nome)

## 01 · Quadro dos parâmetros

In [ ]:
parametros=pd.DataFrame([
    ['Completude','Bytes', '25.000 bytes'],
    ['Verificabilidade','60% referências + 40% densidade','48 refs.; 17,4 refs./10 mil bytes'],
    ['Estrutura','90% seções + 10% infobox','10 seções; presença da infobox'],
    ['Conectividade','50% internos + 20% externos + 20% categorias + 10% Ver também','100; 4; 4; presença da seção'],
    ['Recursos visuais','70% imagens + 30% densidade','3 imagens; 1 imagem/10 mil bytes']
],columns=['Dimensão','Componentes','Referência'])
fig,ax=plt.subplots(figsize=(12,4.1));ax.axis('off')
cells=[[dividir(26)(x) for x in r] for r in parametros.to_numpy()]
t=ax.table(cellText=cells,colLabels=parametros.columns,cellLoc='left',loc='center',
           colWidths=[.17,.46,.37])
t.auto_set_font_size(False);t.set_fontsize(9.4);t.scale(1,2.4)
for (row,col),c in t.get_celld().items():
    c.set_edgecolor('#cbd5e1');c.set_facecolor('#eef4f8' if row%2 else 'white')
    if row==0:c.set_facecolor('#244b70');c.get_text().set_color('white');c.get_text().set_weight('bold')
ax.set_title('Síntese dos indicadores e parâmetros de referência',pad=18,fontweight='bold')
salvar(fig,'01_quadro_parametros.png')

## 02 · Médias Antes × Depois (apenas cinco indicadores)

In [ ]:
val1=[verbetes[f'Antes: {k}'].mean() for k in DIM]
val2=[verbetes[f'Depois: {k}'].mean() for k in DIM]
fig,ax=plt.subplots(figsize=(11,5.6));x=np.arange(5);w=.35
ax.bar(x-w/2,val1,width=w,color=AZUL,label='Antes');ax.bar(x+w/2,val2,width=w,color=LARANJA,label='Depois')
ax.set_xticks(x,['Completude','Verificabilidade','Estrutura e\norganização','Conectividade','Recursos visuais'])
ax.set_ylim(0,1.05);ax.set_ylabel('Pontuação média (0 a 1)');ax.legend(frameon=False)
ax.set_title('Média dos indicadores: antes e depois',pad=15,fontweight='bold');ax.grid(axis='y',alpha=.2)
for xp,vals in [(x-w/2,val1),(x+w/2,val2)]:
    for p,v in zip(xp,vals):ax.text(p,v+.02,f'{v:.2f}',ha='center',fontsize=9)
fig.tight_layout();salvar(fig,'02_indicadores_medios_antes_depois.png')

## 03 · Soma exploratória média (escala própria)

In [ ]:
val=[verbetes['Antes: S'].mean(),verbetes['Depois: S'].mean()]
fig,ax=plt.subplots(figsize=(7,5));ax.bar(['Antes','Depois'],val,color=[AZUL,LARANJA],width=.52)
ax.set_ylim(0,5);ax.set_ylabel('Soma média dos cinco indicadores (0 a 5)');ax.grid(axis='y',alpha=.2)
for i,v in enumerate(val):ax.text(i,v+.08,f'{v:.2f}',ha='center',fontweight='bold')
ax.set_title('Soma exploratória média antes e depois',fontweight='bold');fig.tight_layout()
salvar(fig,'03_soma_exploratoria_media.png')

## 04 · Boxplots dos seis painéis (sem cortes)

In [ ]:
fig,axes=plt.subplots(2,3,figsize=(12.5,8.2))
for ax,(chave,titulo) in zip(axes.flat, list(zip(DIM,NOMES))+[('S','Soma exploratória')]):
    serie=[verbetes[f'Antes: {chave}'],verbetes[f'Depois: {chave}']]
    bp=ax.boxplot(serie,tick_labels=['Antes','Depois'],patch_artist=True,
                  medianprops={'color':'#7a401d','linewidth':1.6},widths=.55)
    for b,c in zip(bp['boxes'],[AZUL,LARANJA]):b.set_facecolor(c);b.set_alpha(.45)
    ax.set_ylim(-.04,5.22 if chave=='S' else 1.09)
    ax.set_title(titulo,fontsize=11,pad=12);ax.grid(axis='y',alpha=.18)
fig.suptitle('Distribuição das pontuações antes e depois',fontweight='bold',fontsize=16,y=.99)
fig.subplots_adjust(top=.88,bottom=.09,hspace=.38,wspace=.25)
salvar(fig,'04_distribuicao_boxplots.png')

## 05 · Evolução por condição inicial

In [ ]:
criados=verbetes['Criado (1=sim)'].eq(1)
val_c=[verbetes.loc[criados,f'Δ Depois−Antes: {k}'].mean() for k in DIM]
val_p=[verbetes.loc[~criados,f'Δ Depois−Antes: {k}'].mean() for k in DIM]
fig,ax=plt.subplots(figsize=(11,5.7));x=np.arange(5);w=.35
ax.bar(x-w/2,val_c,w,color=LARANJA,label='Criados do zero (21)')
ax.bar(x+w/2,val_p,w,color=AZUL,label='Preexistentes (49)')
ax.set_xticks(x,['Completude','Verificabilidade','Estrutura e\norganização','Conectividade','Visuais'])
ax.set_ylim(0,1.05);ax.set_ylabel('Variação média (0 a 1)');ax.legend(frameon=False)
ax.grid(axis='y',alpha=.2);ax.set_title('Evolução dos indicadores conforme a condição inicial',fontweight='bold')
fig.tight_layout();salvar(fig,'05_variacao_por_condicao.png')

## 06 · Top 15 por pontuação final

In [ ]:
qual=verbetes.rename(columns={'Depois: S':'ordem'})
barras_horizontais(qual,'Verbete','ordem','Top 15 verbetes por pontuação final',
                    '06_ranking_qualidade_final.png',cor=VERDE)

## 07 · Top 15 por maior evolução

In [ ]:
ganhos=verbetes.rename(columns={'Δ Depois−Antes: S':'ordem'})
barras_horizontais(ganhos,'Verbete','ordem','Top 15 verbetes por evolução da soma',
                    '07_ranking_evolucao.png',cor=LARANJA)

## 08 · Heatmap dos 15 de maior pontuação final

In [ ]:
top=verbetes.sort_values('Depois: S',ascending=False).head(15)
heatmap(top,'Verbete',[f'Depois: {k}' for k in DIM],
        'Perfil dos 15 verbetes com maior pontuação final','08_heatmap_qualidade_final.png')

## 09 · Heatmap dos 15 de maior evolução

In [ ]:
top=verbetes.sort_values('Δ Depois−Antes: S',ascending=False).head(15)
heatmap(top,'Verbete',[f'Δ Depois−Antes: {k}' for k in DIM],
        'Variação por dimensão nos 15 verbetes que mais evoluíram','09_heatmap_evolucao.png',diferencas=True)

## 10 · Radar dos cinco verbetes com melhor pontuação final

In [ ]:
radar(verbetes,'Verbete',[f'Depois: {k}' for k in DIM],'Depois: S',
      'Radar — Top 5 verbetes por pontuação final','10_radar_top5_verbetes.png')

## 11 · Top 15 participações individuais (aluno + verbete)

In [ ]:
part=individuais.copy();part['rotulo']=part['nome'].astype(str)+' — '+part['verbete'].astype(str)
barras_horizontais(part,'rotulo','S','Top 15 participações individuais (aluno + verbete)',
                    '11_ranking_participacoes_individuais.png',cor=AZUL)

## 12 · Radar das cinco maiores participações individuais

In [ ]:
radar(part,'rotulo',DIM,'S','Radar — Top 5 participações individuais',
      '12_radar_top5_participacoes.png')

## 13 · Top 15 grupos/verbetes por contribuição agregada

In [ ]:
barras_horizontais(grupos,'verbete','S','Top 15 grupos/verbetes por contribuição agregada',
                    '13_ranking_grupos_agregados.png',cor=ROXO)

## 14 · Radar dos cinco grupos/verbetes por contribuição agregada

In [ ]:
radar(grupos,'verbete',DIM,'S','Radar — Top 5 grupos/verbetes por contribuição agregada',
      '14_radar_top5_grupos.png')

## 15 · Medianas e valores de referência em escalas separadas

In [ ]:
metricas=[('Bytes','bytes',25_000),('Referências','referencias',48),
          ('Seções','secoes',10),('Links internos','links_internos',100),
          ('Links externos','links_externos',4),('Categorias','categorias',4),
          ('Imagens','imagens',3)]
fig,axes=plt.subplots(3,3,figsize=(12,10.3))
for ax,(nome,campo,referencia) in zip(axes.flat,metricas):
    dados=[verbetes[f'Antes: {campo}'].median(),verbetes[f'Depois: {campo}'].median(),referencia]
    ax.bar(['Antes','Depois','Referência'],dados,color=[AZUL,LARANJA,CINZA],width=.65)
    ax.set_ylim(0,max(dados)*1.33 if max(dados)>0 else 1)
    ax.set_title(nome,pad=16)
    for i,v in enumerate(dados):ax.text(i,v+max(dados)*.04, f'{v:,.1f}'.replace(',','.') if v%1 else f'{v:,.0f}'.replace(',','.'),ha='center',fontsize=9)
    ax.grid(axis='y',alpha=.15)
for ax in axes.flat[len(metricas):]:ax.axis('off')
fig.suptitle('Medianas observadas e valores de referência por métrica',fontweight='bold',y=.985)
fig.subplots_adjust(top=.91,bottom=.06,hspace=.64,wspace=.28)
salvar(fig,'15_medianas_por_metrica.png')

## 16 · Pontuação inicial versus evolução com tendência

In [ ]:
x=verbetes['Antes: S'].to_numpy();y=verbetes['Δ Depois−Antes: S'].to_numpy()
e=verbetes['Criado (1=sim)'].eq(0).to_numpy()
fig,ax=plt.subplots(figsize=(9,6))
ax.scatter(x[e],y[e],color=AZUL,label='Preexistentes',alpha=.83)
ax.scatter(x[~e],y[~e],color=LARANJA,label='Criados',alpha=.83)
para_x=np.linspace(x.min(),x.max(),150)
coef=np.polyfit(x,y,1)
ax.plot(para_x,np.polyval(coef,para_x),color=ROXO,lw=2,linestyle='--',label='Tendência geral')
coef_p=np.polyfit(x[e],y[e],1)
ax.plot(para_x,np.polyval(coef_p,para_x),color=VERDE,lw=2,label='Tendência nos preexistentes')
ax.set_xlabel('Pontuação inicial (0 a 5)');ax.set_ylabel('Evolução (depois − antes)')
ax.set_title('Pontuação inicial versus evolução dos verbetes',fontweight='bold')
ax.legend(frameon=False,fontsize=9);ax.grid(alpha=.18);fig.tight_layout()
salvar(fig,'16_pontuacao_inicial_evolucao_tendencia.png')

## 17 · Todos os 21 verbetes criados do zero

In [ ]:
criados=verbetes.loc[verbetes['Criado (1=sim)'].eq(1)].copy()
assert len(criados)==21
barras_horizontais(criados,'Verbete','Depois: S',
                    '21 verbetes criados do zero — pontuação final',
                    '17_todos_21_criados_do_zero.png',cor=AZUL)

## 18 e 19 · Preexistentes: base inicial e acréscimo acadêmico

Nesta comparação, **não** somamos a pontuação independente do grupo à pontuação inicial. Acrescentamos os saldos positivos das métricas dos estudantes ao estado inicial, recalculamos os cinco indicadores e, só então, obtemos o ganho. O resultado respeita o teto de 5 e é uma estimativa baseada nas edições acadêmicas, não o estado Depois observado com terceiros.

In [ ]:
METRICAS=['bytes','referencias','secoes','infobox','ver_tambem','links_internos','links_externos','categorias','imagens']
def score(m):
    b,r,s,inf,vt,li,le,c,im=[max(0,float(m.get(k,0) or 0)) for k in METRICAS]
    dr=10_000*r/b if b else 0;di=10_000*im/b if b else 0
    cap=lambda x:max(0,min(1,x))
    return (cap(b/25_000)+.6*cap(r/48)+.4*cap(dr/17.4)+
            .9*cap(s/10)+.1*cap(inf)+.5*cap(li/100)+.2*cap(le/4)+
            .2*cap(c/4)+.1*cap(vt)+.7*cap(im/3)+.3*cap(di))

j=verbetes.loc[verbetes['Criado (1=sim)'].eq(0)].merge(
    grupos[['id']+[f'Saldo bruto: {k}' for k in METRICAS]],
    left_on='ID unidade',right_on='id',validate='one_to_one')
linhas=[]
for _,r in j.iterrows():
    anterior={k:r[f'Antes: {k}'] for k in METRICAS}
    combinado={k:anterior[k]+max(0,r[f'Saldo bruto: {k}']) for k in METRICAS}
    inicio=score(anterior);final=score(combinado)
    linhas.append({'Verbete':r['Verbete'],'Antes':inicio,
                   'Acréscimo':final-inicio,'Final estimado':final})
pre=pd.DataFrame(linhas)
assert pre['Final estimado'].between(0,5).all()

def barra_acrescimo(coluna_ordem,nome,titulo):
    top=pre.nlargest(20,coluna_ordem).sort_values(coluna_ordem,ascending=True)
    assert (top['Acréscimo']>=-1e-8).all(), 'Um caso com perda requer outra representação.'
    fig,ax=plt.subplots(figsize=(12,13))
    y=np.arange(len(top));base=top['Antes'].to_numpy();ganho=top['Acréscimo'].to_numpy()
    ax.barh(y,base,color=AZUL,height=.65,label='Pontuação antes')
    ax.barh(y,ganho,left=base,color=LARANJA,height=.65,label='Ganho reestimado dos alunos')
    ax.set_yticks(y,[dividir(31)(r) for r in top['Verbete']],fontsize=9.5)
    ax.set_xlim(0,5);ax.set_xlabel('Soma dos cinco indicadores (0 a 5)')
    ax.set_title(titulo,pad=15,fontweight='bold')
    for i,t in enumerate(top['Final estimado']):
        ax.text(min(4.99,t+.035),i,f'{t:.2f}',ha='left' if t<4.7 else 'right',va='center',fontsize=9)
    ax.legend(loc='lower right',frameon=False);ax.grid(axis='x',alpha=.18)
    ax.tick_params(axis='y',length=0)
    fig.subplots_adjust(left=.35,right=.97,top=.94,bottom=.08)
    salvar(fig,nome)

## 18 · Top 20 preexistentes por maior ganho do grupo

In [ ]:
barra_acrescimo('Acréscimo','18_preexistentes_top20_maior_ganho.png',
                'Preexistentes: Top 20 por ganho reestimado dos alunos')

## 19 · Top 20 preexistentes por pontuação final estimada

In [ ]:
barra_acrescimo('Final estimado','19_preexistentes_top20_final_estimado.png',
                'Preexistentes: Top 20 por pontuação final reestimada')